# 🫁 CheXpert Medical AI - Training & Optimization on Google Colab GPU

Notebook này giúp bạn huấn luyện mô hình phân tích ảnh X-quang phổi CheXpert đạt độ chính xác cao nhất (State-of-the-Art) trên GPU của Google Colab.

### Các cải tiến nổi bật:
1. **Đa kiến trúc hiện đại**: `convnext_small`, `efficientnet_v2_m`, `densenet121`, `resnet50`.
2. **Asymmetric Loss (ASL) / Focal Loss**: Chống mất cân bằng nhãn bệnh lý cực tốt.
3. **Stanford U-Ones Policy**: Xử lý nhãn nghi ngờ (`-1`) chuẩn xác để tối đa hóa độ nhạy (Sensitivity).
4. **Threshold Calibration**: Tự động tìm ngưỡng xác suất tối ưu F1-score riêng cho từng bệnh lý.
5. **Mixed Precision (AMP)**: Tăng tốc độ huấn luyện gấp 2-3 lần trên GPU.

## 1. Kiểm tra GPU Colab

In [ ]:
!nvidia-smi

## 2. Clone dự án & Cài đặt môi trường

In [ ]:
!git clone https://github.com/qdat2644/chex.git
%cd chex
!pip install -q -r requirements.txt
!pip install -q kaggle

## 3. Tải bộ dữ liệu CheXpert (qua Kaggle API hoặc Google Drive)

Nếu bạn dùng Kaggle:
1. Vào [Kaggle Account](https://www.kaggle.com/settings) -> bấm **Create New Token** để tải file `kaggle.json`.
2. Chạy cell dưới đây và upload file `kaggle.json` lên Colab.

In [ ]:
from google.colab import files
import os

print("Upload file kaggle.json của bạn:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải dataset CheXpert (bản Kaggle 11GB nén)
!kaggle datasets download -d ashery/chexpert -p archive/ --unzip

## 4. Huấn luyện Mô hình Tối ưu (Training)

Bạn có thể chọn 1 trong các kiến trúc sau:
- `convnext_small` (Khuyên dùng: Hiện đại, chính xác cao)
- `efficientnet_v2_m` (Cân bằng tốt giữa tốc độ và độ chính xác)
- `densenet121` (Chuẩn mực CheXpert gốc)

In [ ]:
# Lệnh huấn luyện ConvNeXt-Small với Asymmetric Loss & Stanford U-Ones
!python scripts/train.py \
    --data-root archive \
    --arch convnext_small \
    --loss asl \
    --image-size 224 \
    --epochs 6 \
    --batch-size 32 \
    --lr 1e-4 \
    --uncertain-policy u_ones_zeros \
    --scheduler cosine \
    --pretrained \
    --amp \
    --output checkpoints/chexpert_convnext_small.pt

## 5. Đánh giá & Hiệu chuẩn Ngưỡng chẩn đoán (Threshold Calibration)

In [ ]:
!python scripts/evaluate.py \
    --data-root archive \
    --checkpoint checkpoints/chexpert_convnext_small.pt \
    --output-thresholds outputs/evaluation/thresholds.json

## 6. Tải Model & File Thresholds về máy tính

In [ ]:
from google.colab import files

print("Tải file model checkpoint (.pt):")
files.download('checkpoints/chexpert_convnext_small.pt')

print("Tải file ngưỡng phân loại (thresholds.json):")
files.download('outputs/evaluation/thresholds.json')

print("Hoàn tất! Hãy copy 2 file này vào dự án trên máy local để trải nghiệm model mới!")